[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/02_multimodal_beyond_vision.ipynb)

# 02. Beyond Vision: Audio + Text + Video

**This notebook covers:**
- Audio representations (spectrograms, Whisper embeddings)
- Audio-Text models (CLAP)
- Video understanding (temporal dimension)
- ImageBind: one embedding space for 6 modalities
- Building an audio-image-text model from scratch

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/05_Advanced_Topics")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# The multimodal landscape beyond vision+text

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('The Full Multimodal Landscape', fontsize=18, fontweight='bold', pad=20)

modalities = [
    (2, 7, 'Image\n(pixels)', '#E74C3C'),
    (5, 7, 'Text\n(tokens)', '#3498DB'),
    (8, 7, 'Audio\n(waveform)', '#2ECC71'),
    (11, 7, 'Video\n(frames)', '#F39C12'),
    (3.5, 5.5, 'Depth\n(3D)', '#9B59B6'),
    (10, 5.5, 'IMU/Sensor\n(motion)', '#1ABC9C'),
]

for x, y, label, color in modalities:
    draw_architecture_block(ax, x, y, 2.5, 0.8, label, color)

draw_architecture_block(ax, 7, 3.5, 10, 1.2, 'Shared Embedding Space\n(e.g., ImageBind: ONE space for ALL modalities)', '#34495E', fontsize=11)

for x, y, _, _ in modalities:
    draw_arrow(ax, (x, y-0.5), (7 + (x-7)*0.3, 4.2))

tasks = [
    (3, 1.5, 'Cross-modal\nRetrieval'),
    (7, 1.5, 'Zero-shot\nClassification'),
    (11, 1.5, 'Any-to-Any\nGeneration'),
]
for x, y, label in tasks:
    draw_architecture_block(ax, x, y, 3, 0.7, label, '#7F8C8D')
    draw_arrow(ax, (x, 2.8), (x, 2.0))

plt.tight_layout()
plt.savefig('../assets/full_multimodal.png', dpi=150, bbox_inches='tight')
plt.show()

## 1. Audio Representations

In [ ]:
# Visualize: How audio becomes embeddings

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Audio Processing Pipeline', fontsize=16, fontweight='bold')

# 1. Waveform
ax = axes[0]
t = np.linspace(0, 1, 1000)
waveform = np.sin(2 * np.pi * 440 * t) * np.exp(-2*t) + np.random.randn(1000) * 0.1
ax.plot(t, waveform, color='#2ECC71', linewidth=0.5)
ax.set_title('1. Raw Waveform')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')

# 2. Spectrogram
ax = axes[1]
spec = np.random.rand(64, 100) ** 2
# Add some structure
for f in [10, 20, 40]:
    spec[f-2:f+2, :] += np.random.rand(4, 100) * 2
ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
ax.set_title('2. Mel Spectrogram')
ax.set_xlabel('Time frames')
ax.set_ylabel('Mel frequency bins')

# 3. Patch embedding (like ViT for audio)
ax = axes[2]
patches = np.random.randn(16, 32) * 0.5
ax.imshow(patches, aspect='auto', cmap='RdBu_r')
ax.set_title('3. Patch Embeddings\n(spectrogram patches)')
ax.set_xlabel('Embedding dim (first 32)')
ax.set_ylabel('Patch index')

# 4. Output embedding
ax = axes[3]
embedding = np.random.randn(64)
ax.bar(range(64), embedding, color='#2ECC71', alpha=0.7)
ax.set_title('4. Audio Embedding\n(fixed-size vector)')
ax.set_xlabel('Dimension')
ax.set_ylabel('Value')

plt.tight_layout()
plt.savefig('../assets/audio_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build a simple audio encoder

class SimpleAudioEncoder(nn.Module):
    """Encodes a mel spectrogram into an embedding vector."""
    def __init__(self, n_mels=64, embed_dim=128, n_layers=2):
        super().__init__()
        # Treat spectrogram as 1-channel image
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, embed_dim, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.flatten_dim = embed_dim * 4 * 4
        self.proj = nn.Linear(self.flatten_dim, embed_dim)

    def forward(self, mel_spec):
        # mel_spec: [B, 1, n_mels, T]
        x = self.conv(mel_spec).flatten(1)
        return self.proj(x)


audio_enc = SimpleAudioEncoder(embed_dim=128)
dummy_mel = torch.randn(2, 1, 64, 100)  # batch=2, 64 mel bins, 100 time frames
audio_emb = audio_enc(dummy_mel)
print(f"Audio input: {dummy_mel.shape}")
print(f"Audio embedding: {audio_emb.shape}")
count_parameters(audio_enc)

## 2. Three-Modal Model: Image + Text + Audio

In [ ]:
class ThreeModalCLIP(nn.Module):
    """CLIP-style model for Image + Text + Audio."""
    def __init__(self, embed_dim=128, proj_dim=64):
        super().__init__()
        # Image encoder (small CNN)
        self.img_enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # Text encoder
        self.txt_enc = nn.Sequential(
            nn.Embedding(1000, embed_dim),
        )
        # Audio encoder
        self.aud_enc = SimpleAudioEncoder(embed_dim=embed_dim)

        # Shared projection heads
        self.img_proj = nn.Linear(embed_dim, proj_dim)
        self.txt_proj = nn.Linear(embed_dim, proj_dim)
        self.aud_proj = nn.Linear(embed_dim, proj_dim)

        self.temperature = nn.Parameter(torch.ones(1) * 0.07)

    def encode(self, images=None, text_ids=None, audio=None):
        outputs = {}
        if images is not None:
            ie = F.normalize(self.img_proj(self.img_enc(images)), dim=-1)
            outputs['image'] = ie
        if text_ids is not None:
            te = self.txt_enc(text_ids).mean(dim=1)
            te = F.normalize(self.txt_proj(te), dim=-1)
            outputs['text'] = te
        if audio is not None:
            ae = F.normalize(self.aud_proj(self.aud_enc(audio)), dim=-1)
            outputs['audio'] = ae
        return outputs


model = ThreeModalCLIP()
count_parameters(model)

# Test all modalities
embs = model.encode(
    images=torch.randn(4, 3, 32, 32),
    text_ids=torch.randint(0, 1000, (4, 10)),
    audio=torch.randn(4, 1, 64, 100)
)
for name, emb in embs.items():
    print(f"{name:6s} embedding: {emb.shape}")

# Cross-modal similarity
sim_img_txt = (embs['image'] @ embs['text'].T).detach()
sim_img_aud = (embs['image'] @ embs['audio'].T).detach()
sim_txt_aud = (embs['text'] @ embs['audio'].T).detach()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sim, title in zip(axes, [sim_img_txt, sim_img_aud, sim_txt_aud],
                          ['Image ↔ Text', 'Image ↔ Audio', 'Text ↔ Audio']):
    ax.imshow(sim.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(title, fontsize=12, fontweight='bold')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=9)

fig.suptitle('Cross-Modal Similarities (untrained)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Video Understanding

Video = Image + **temporal dimension**. Key approaches:

| Approach | How | Example |
|----------|-----|---------|
| Frame sampling | Encode N frames independently | VideoCLIP |
| Temporal attention | Add time attention layers | TimeSformer |
| 3D convolution | Conv across space + time | C3D, SlowFast |
| Token merging | Merge frame tokens | Video-LLaVA |

In [ ]:
class SimpleVideoEncoder(nn.Module):
    """Encode video by sampling frames + temporal attention."""
    def __init__(self, embed_dim=128, n_frames=8):
        super().__init__()
        self.n_frames = n_frames
        # Per-frame encoder
        self.frame_enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # Temporal attention
        self.temporal_pos = nn.Embedding(n_frames, embed_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.temporal_attn = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, frames):
        """frames: [B, n_frames, C, H, W]"""
        B, T, C, H, W = frames.shape
        # Encode each frame
        frame_feats = self.frame_enc(frames.view(B*T, C, H, W))  # [B*T, D]
        frame_feats = frame_feats.view(B, T, -1)                  # [B, T, D]
        # Add temporal position
        pos = self.temporal_pos(torch.arange(T, device=frames.device))
        frame_feats = frame_feats + pos
        # Temporal attention
        output = self.temporal_attn(frame_feats)
        return self.norm(output.mean(dim=1))  # [B, D] pool over time


video_enc = SimpleVideoEncoder(embed_dim=128, n_frames=8)
dummy_video = torch.randn(2, 8, 3, 32, 32)  # 2 videos, 8 frames
video_emb = video_enc(dummy_video)
print(f"Video input:     {dummy_video.shape} (batch, frames, C, H, W)")
print(f"Video embedding: {video_emb.shape}")
count_parameters(video_enc)

## Key Takeaways

1. **Audio** → mel spectrogram → treat like an image → encode with ViT/CNN
2. **Video** → sample frames → encode each → temporal attention
3. **ImageBind** proved you can align 6 modalities in ONE space using image as the anchor
4. The pattern is always: **Modality Encoder → Projection → Shared Space**
5. **Contrastive learning** works across any pair of modalities

---
**Next:** `03_efficient_deployment.ipynb` — Ship your model efficiently